In [ ]:
# # Preprocessing script for 25_tags_events dataset

# import glob
# import pandas as pd
# import numpy as np
# from pathlib import Path

# # ── Configuration ───────────────────────────────────────────────────
# PROJECT_ROOT = Path('/home/h604827/ControlActions')
# INPUT_DIR  = PROJECT_ROOT / 'DATA/25_tags_events'
# OUTPUT_DIR = PROJECT_ROOT / 'DATA/25_tags_events_preprocessed'
# TRIP_CSV   = PROJECT_ROOT / 'DATA/Final_List_Trip_Duration.csv'
# TIME_OFFSET = pd.Timedelta(hours=1.5)
# DEDUP_COLS  = ['VT_Start', 'Source', 'ConditionName', 'Description']
# CHUNK_SIZE  = 100_000

# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# # ── Load trip periods (once) ────────────────────────────────────────
# trip_data = pd.read_csv(TRIP_CSV)
# trip_data['Stop Date']  = pd.to_datetime(trip_data['Stop Date'])
# trip_data['Start Date'] = pd.to_datetime(trip_data['Start Date'])
# stop_dates  = trip_data['Stop Date'].values
# start_dates = trip_data['Start Date'].values

# # ── Discover UUID folders ──────────────────────────────────────────
# uuid_dirs = sorted([d for d in INPUT_DIR.iterdir() if d.is_dir()])
# print(f"Found {len(uuid_dirs)} UUID folders in {INPUT_DIR}")

# # ── Process each UUID folder ───────────────────────────────────────
# summary = []

# for uuid_dir in uuid_dirs:
#     uuid_name = uuid_dir.name

#     # Find all _E.parquet part files under this UUID folder
#     parquet_files = glob.glob(
#         str(uuid_dir / '2021011814_2025062803_E.parquet' / 'S=*' / '*.parquet')
#     )
#     if not parquet_files:
#         print(f"  [{uuid_name}] No _E parquet files found – skipping")
#         continue

#     # 1. Read and concatenate all part files
#     dfs = [pd.read_parquet(f) for f in parquet_files]
#     df = pd.concat(dfs, ignore_index=True)
#     n_raw = len(df)

#     # 2. Adjust VT_Start by -1.5 hours
#     if 'VT_Start' in df.columns:
#         df['VT_Start'] = pd.to_datetime(df['VT_Start'], format='mixed') - TIME_OFFSET
#     else:
#         print(f"  [{uuid_name}] WARNING: VT_Start column not found – skipping time offset")

#     df = df.sort_values('VT_Start').reset_index(drop=True)

#     # 3. Remove events during trip periods (chunked broadcasting)
#     event_times = df['VT_Start'].values
#     in_trips = np.zeros(len(event_times), dtype=bool)

#     for i in range(0, len(event_times), CHUNK_SIZE):
#         chunk = event_times[i:i + CHUNK_SIZE]
#         in_range = (chunk[:, None] >= stop_dates) & (chunk[:, None] <= start_dates)
#         in_trips[i:i + CHUNK_SIZE] = in_range.any(axis=1)

#     n_trip = int(in_trips.sum())
#     df = df[~in_trips].reset_index(drop=True)

#     # 4. Deduplicate rows
#     n_pre_dedup = len(df)
#     df = df.groupby(DEDUP_COLS, sort=False).first().reset_index()
#     df = df.sort_values('VT_Start').reset_index(drop=True)
#     n_deduped = n_pre_dedup - len(df)

#     # 5. Save preprocessed parquet
#     out_path = OUTPUT_DIR / f'{uuid_name}.parquet'
#     df.to_parquet(out_path, index=False)

#     summary.append({
#         'uuid': uuid_name,
#         'raw_rows': n_raw,
#         'trip_removed': n_trip,
#         'duplicates_removed': n_deduped,
#         'final_rows': len(df),
#         'sources': df['Source'].nunique() if 'Source' in df.columns else None
#     })
#     print(f"  [{uuid_name}] {n_raw} → {len(df)} rows "
#           f"(trips: -{n_trip}, dupes: -{n_deduped}) → {out_path.name}")

# # ── Summary ────────────────────────────────────────────────────────
# summary_df = pd.DataFrame(summary)
# print(f"\n{'='*70}")
# print(f"Preprocessed {len(summary_df)} UUID folders → {OUTPUT_DIR}/")
# print(f"Total: {summary_df['raw_rows'].sum():,} raw → {summary_df['final_rows'].sum():,} final rows")
# print(summary_df.to_string(index=False))

Found 11 UUID folders in /home/h604827/ControlActions/DATA/25_tags_events
  [0102a4fc-aae1-414b-a790-e62529880ede] 110212 → 54540 rows (trips: -11197, dupes: -44475) → 0102a4fc-aae1-414b-a790-e62529880ede.parquet
  [032761f5-dbdf-4d83-9566-498b49fa05ee] 220374 → 124600 rows (trips: -11290, dupes: -84484) → 032761f5-dbdf-4d83-9566-498b49fa05ee.parquet
  [3ad21a42-9b2f-450a-91f3-f78b9cb4e99e] 91578 → 31166 rows (trips: -18806, dupes: -41606) → 3ad21a42-9b2f-450a-91f3-f78b9cb4e99e.parquet
  [572ec917-c98e-4932-9cc6-eacd831be962] 124569 → 68510 rows (trips: -5070, dupes: -50989) → 572ec917-c98e-4932-9cc6-eacd831be962.parquet
  [6b9de746-f4ab-4082-acfa-75bf21530bbf] 1051748 → 906230 rows (trips: -96538, dupes: -48980) → 6b9de746-f4ab-4082-acfa-75bf21530bbf.parquet
  [8c7c9aeb-3c57-4e87-8ee0-5e33c0499e8d] 523768 → 313410 rows (trips: -142979, dupes: -67379) → 8c7c9aeb-3c57-4e87-8ee0-5e33c0499e8d.parquet
  [a74a1df6-3231-4c91-be6a-3179156c1572] 24298 → 14467 rows (trips: -5419, dupes: -4412) 

In [14]:
data_path = '/home/h604827/ControlActions/DATA/25_tags_events_preprocessed'

alarm_tags = [
    ['03PIC_1620', 'PVHI'],
    ['03FIC_1668', 'PVHI'],
    ['03PI_1655', 'PVHI'],
    ['03TIC_1635', 'PVHI'],
    ['03TIC_1009', 'PVHI'],
    ['03TIC_1745A', 'PVHI'],
    ['03LIC_1608', 'PVHI'],
    ['03PIC_1013', 'PVLO'],
    ['03LIC_1608', 'PVLO'],
    ['03PIC_1023', 'PVLO'],
    ['03TIC_1145', 'PVLO'],
    ['03LIC_1071', 'PVHI'],
    ['03LIC_1016', 'PVHI'],
    ['03PIC_1104', 'PVHI'],
    ['03TIC_1009', 'PVLO'],
    ['03LIC_1619', 'PVLO'],
    ['03TIC_1635', 'PVLO'],
    ['03PIC_1023', 'PVHI'],
    ['03LIC_1619', 'PVHI'],
    ['03PI_1814', 'PVHI'],
    ['03TIC_1023', 'PVLO'],
    ['03TIC_1023', 'PVHI'],
    ['03LIC_1016', 'PVLO'],
    ['03LIC_1071', 'PVLO'],
    ['03TI_1081', 'PVHI'],
]

In [15]:
import pandas as pd
from pathlib import Path

data_path = Path('/home/h604827/ControlActions/DATA/25_tags_events_preprocessed')
parquet_files = sorted(data_path.glob('*.parquet'))
events_csv_path = '/home/h604827/ControlActions/DATA/trip_filtered_events_dedup.csv'

# For each parquet file, check which alarm_tag entries it contains
results = []
for pf in parquet_files:
    df_tmp = pd.read_parquet(pf, columns=['Source', 'ConditionName'])
    for tag, condition in alarm_tags:
        mask = (df_tmp['Source'] == tag) & (df_tmp['ConditionName'] == condition)
        count = mask.sum()
        if count > 0:
            results.append({'file': pf.name, 'tag': tag, 'condition': condition, 'event_count': count})

# Check trip_filtered_events_dedup.csv for remaining tags
events_df = pd.read_csv(events_csv_path, low_memory=False)
for tag, condition in alarm_tags:
    mask = (events_df['Source'] == tag) & (events_df['ConditionName'] == condition)
    count = mask.sum()
    if count > 0:
        results.append({'file': 'trip_filtered_events_dedup.csv', 'tag': tag, 'condition': condition, 'event_count': count})

results_df = pd.DataFrame(results)
print(f"Total matches: {len(results_df)}")
print(f"\nAlarm tags with matches: {results_df[['tag','condition']].drop_duplicates().shape[0]} / {len(alarm_tags)}")
print()
results_df.pivot_table(index=['tag', 'condition'], columns='file', values='event_count', fill_value=0)

Total matches: 32

Alarm tags with matches: 25 / 25



file                   572ec917-c98e-4932-9cc6-eacd831be962.parquet  \
tag         condition                                                 
03FIC_1668  PVHI                                                0.0   
03LIC_1016  PVHI                                                0.0   
            PVLO                                                0.0   
03LIC_1071  PVHI                                                0.0   
            PVLO                                                0.0   
03LIC_1608  PVHI                                                0.0   
            PVLO                                                0.0   
03LIC_1619  PVHI                                                0.0   
            PVLO                                                0.0   
03PIC_1013  PVLO                                                0.0   
03PIC_1023  PVHI                                                0.0   
            PVLO                                                0.0   
03PIC_1104  PVHI                                                0.0   
03PIC_1620  PVHI                                                0.0   
03PI_1655   PVHI                                                0.0   
03PI_1814   PVHI                                                0.0   
03TIC_1009  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1023  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1145  PVLO                                                0.0   
03TIC_1635  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1745A PVHI                                               31.0   
03TI_1081   PVHI                                                0.0   

file                   6b9de746-f4ab-4082-acfa-75bf21530bbf.parquet  \
tag         condition                                                 
03FIC_1668  PVHI                                                0.0   
03LIC_1016  PVHI                                                0.0   
            PVLO                                                0.0   
03LIC_1071  PVHI                                                0.0   
            PVLO                                                0.0   
03LIC_1608  PVHI                                                0.0   
            PVLO                                                0.0   
03LIC_1619  PVHI                                                0.0   
            PVLO                                                0.0   
03PIC_1013  PVLO                                               76.0   
03PIC_1023  PVHI                                                0.0   
            PVLO                                                0.0   
03PIC_1104  PVHI                                                0.0   
03PIC_1620  PVHI                                                0.0   
03PI_1655   PVHI                                                0.0   
03PI_1814   PVHI                                              900.0   
03TIC_1009  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1023  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1145  PVLO                                              228.0   
03TIC_1635  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1745A PVHI                                                0.0   
03TI_1081   PVHI                                                0.0   

file                   c947bc21-1c5c-4240-aa1c-ba53ebfc5b7c.parquet  \
tag         condition                                                 
03FIC_1668  PVHI      

In [16]:
# Compact view: which file contains each alarm tag
tag_to_file = results_df.groupby(['tag', 'condition']).agg(
    files=('file', list),
    total_events=('event_count', 'sum')
).reset_index()
print(f"Matched: {len(tag_to_file)} / {len(alarm_tags)} alarm tag entries\n")
for _, row in tag_to_file.iterrows():
    print(f"  {row['tag']:15s} | {row['condition']:4s} | events={row['total_events']:5d} | files: {row['files']}")

# Show unmatched tags
matched_pairs = set(zip(tag_to_file['tag'], tag_to_file['condition']))
unmatched = [(t, c) for t, c in alarm_tags if (t, c) not in matched_pairs]
if unmatched:
    print(f"\nUnmatched ({len(unmatched)}):")
    for t, c in unmatched:
        print(f"  {t:15s} | {c}")

Matched: 25 / 25 alarm tag entries

  03FIC_1668      | PVHI | events=   19 | files: ['c947bc21-1c5c-4240-aa1c-ba53ebfc5b7c.parquet']
  03LIC_1016      | PVHI | events=  504 | files: ['trip_filtered_events_dedup.csv']
  03LIC_1016      | PVLO | events= 6607 | files: ['trip_filtered_events_dedup.csv']
  03LIC_1071      | PVHI | events=  998 | files: ['ec444132-ab14-4ed8-b112-63c57bef88ea.parquet', 'trip_filtered_events_dedup.csv']
  03LIC_1071      | PVLO | events= 9036 | files: ['ec444132-ab14-4ed8-b112-63c57bef88ea.parquet', 'trip_filtered_events_dedup.csv']
  03LIC_1608      | PVHI | events=   22 | files: ['c947bc21-1c5c-4240-aa1c-ba53ebfc5b7c.parquet']
  03LIC_1608      | PVLO | events=   63 | files: ['c947bc21-1c5c-4240-aa1c-ba53ebfc5b7c.parquet']
  03LIC_1619      | PVHI | events= 2550 | files: ['c947bc21-1c5c-4240-aa1c-ba53ebfc5b7c.parquet']
  03LIC_1619      | PVLO | events= 2027 | files: ['c947bc21-1c5c-4240-aa1c-ba53ebfc5b7c.parquet']
  03PIC_1013      | PVLO | events=  152 | 

In [18]:
import pandas as pd
from pathlib import Path

data_path = Path('/home/h604827/ControlActions/DATA/25_tags_events_preprocessed')
events_csv_path = '/home/h604827/ControlActions/DATA/trip_filtered_events_dedup.csv'

# Load CSV (already in memory as events_df, but ensure it's available)
events_df = pd.read_csv(events_csv_path, low_memory=False)
events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])

# Build a lookup: which parquet file has each tag
parquet_files = sorted(data_path.glob('*.parquet'))
tag_parquet_map = {}  # (tag, condition) -> parquet file path
for pf in parquet_files:
    df_tmp = pd.read_parquet(pf, columns=['Source', 'ConditionName'])
    sources = df_tmp['Source'].unique()
    conditions = df_tmp['ConditionName'].unique()
    for tag, condition in alarm_tags:
        if tag in sources:
            # Verify this condition exists in the file
            mask = (df_tmp['Source'] == tag) & (df_tmp['ConditionName'] == condition)
            if mask.any():
                tag_parquet_map[(tag, condition)] = pf

def load_tag_events(tag, condition):
    """Load alarm-relevant events for a tag/condition from best source."""
    if (tag, condition) in tag_parquet_map:
        pf = tag_parquet_map[(tag, condition)]
        df = pd.read_parquet(pf)
        df['VT_Start'] = pd.to_datetime(df['VT_Start'])
        df = df[(df['Source'] == tag) & (df['ConditionName'] == condition)]
    else:
        df = events_df[(events_df['Source'] == tag) & (events_df['ConditionName'] == condition)].copy()
    
    # Filter Category == 1
    df = df[df['Category'] == 1].copy()
    df = df.sort_values('VT_Start').reset_index(drop=True)
    return df

def extract_alarms(tag, condition):
    """Extract alarm episodes (start→end pairs) and log orphans."""
    df = load_tag_events(tag, condition)
    
    # Identify starts and ends
    is_start = df['Action'].isna() | (df['Action'] == '')
    is_end = df['Action'] == 'OK'
    
    starts = df[is_start].reset_index(drop=True)
    ends = df[is_end].reset_index(drop=True)
    
    alarms = []
    orphan_starts = []
    orphan_ends = []
    
    # Sequential pairing: walk through events in time order
    events_ordered = df[is_start | is_end].copy()
    events_ordered['is_start'] = events_ordered['Action'].isna() | (events_ordered['Action'] == '')
    events_ordered = events_ordered.sort_values('VT_Start').reset_index(drop=True)
    
    pending_start = None
    for idx, row in events_ordered.iterrows():
        if row['is_start']:
            if pending_start is not None:
                # Previous start had no end → orphan start
                orphan_starts.append(pending_start)
            pending_start = row
        else:  # is_end (Action == 'OK')
            if pending_start is not None:
                # Matched pair
                alarms.append({
                    'tag': tag,
                    'condition': condition,
                    'alarm_start': pending_start['VT_Start'],
                    'alarm_end': row['VT_Start'],
                    'duration_minutes': (row['VT_Start'] - pending_start['VT_Start']).total_seconds() / 60
                })
                pending_start = None
            else:
                # End with no preceding start → orphan end
                orphan_ends.append(row)
    
    # If there's a remaining pending start at the end
    if pending_start is not None:
        orphan_starts.append(pending_start)
    
    return alarms, orphan_starts, orphan_ends

# ── Extract alarms for all 25 entries ──────────────────────────────
all_alarms = []
all_orphan_starts = []
all_orphan_ends = []

print(f"{'Tag':<15} | {'Cond':<4} | {'Alarms':>6} | {'Orphan Starts':>13} | {'Orphan Ends':>11}")
print("-" * 70)

for tag, condition in alarm_tags:
    alarms, orphan_starts, orphan_ends = extract_alarms(tag, condition)
    all_alarms.extend(alarms)
    
    for os_row in orphan_starts:
        all_orphan_starts.append({'tag': tag, 'condition': condition, 'timestamp': os_row['VT_Start']})
    for oe_row in orphan_ends:
        all_orphan_ends.append({'tag': tag, 'condition': condition, 'timestamp': oe_row['VT_Start']})
    
    print(f"{tag:<15} | {condition:<4} | {len(alarms):>6} | {len(orphan_starts):>13} | {len(orphan_ends):>11}")

print("-" * 70)
print(f"{'TOTAL':<15} | {'':4} | {len(all_alarms):>6} | {len(all_orphan_starts):>13} | {len(all_orphan_ends):>11}")

# Create DataFrames
alarms_df = pd.DataFrame(all_alarms)
orphan_starts_df = pd.DataFrame(all_orphan_starts) if all_orphan_starts else pd.DataFrame(columns=['tag','condition','timestamp'])
orphan_ends_df = pd.DataFrame(all_orphan_ends) if all_orphan_ends else pd.DataFrame(columns=['tag','condition','timestamp'])

print(f"\n\nAlarms DataFrame shape: {alarms_df.shape}")
print(f"Orphan starts: {len(orphan_starts_df)}")
print(f"Orphan ends: {len(orphan_ends_df)}")
alarms_df.head(10)

Tag             | Cond | Alarms | Orphan Starts | Orphan Ends
----------------------------------------------------------------------
03PIC_1620      | PVHI |      6 |             0 |           0
03FIC_1668      | PVHI |      7 |             0 |           0
03PI_1655       | PVHI |      9 |             0 |           0
03TIC_1635      | PVHI |      5 |             0 |           0
03TIC_1009      | PVHI |      3 |             0 |           0
03TIC_1745A     | PVHI |      9 |             0 |           0
03LIC_1608      | PVHI |      7 |             0 |           0
03PIC_1013      | PVLO |     23 |             3 |           0
03LIC_1608      | PVLO |     19 |             0 |           0
03PIC_1023      | PVLO |     18 |             5 |           0
03TIC_1145      | PVLO |     67 |             2 |           0
03LIC_1071      | PVHI |    184 |             1 |           1
03LIC_1016      | PVHI |    154 |             0 |           0
03PIC_1104      | PVHI |    276 |             0 |           0

,tag,condition,alarm_start,alarm_end,duration_minutes
0,03PIC_1620,PVHI,2021-10-15 07:10:19.205900,2021-10-15 07:10:37.205900,0.300000
1,03PIC_1620,PVHI,2021-11-30 09:12:19.601600,2021-11-30 09:13:21.603000,1.033357
2,03PIC_1620,PVHI,2021-12-07 03:21:49.752900,2021-12-07 03:22:21.753600,0.533345
3,03PIC_1620,PVHI,2021-12-07 13:29:57.302900,2021-12-07 13:31:36.307300,1.650073
4,03PIC_1620,PVHI,2023-01-30 08:35:25.603500,2023-01-30 08:36:17.603000,0.866658
5,03PIC_1620,PVHI,2023-11-06 00:50:50.203600,2023-11-06 00:51:34.202700,0.733318
6,03FIC_1668,PVHI,2021-12-07 02:28:58.902700,2021-12-07 02:29:05.903600,0.116682
7,03FIC_1668,PVHI,2024-01-06 10:44:02.103500,2024-01-06 10:44:40.102600,0.633318
8,03FIC_1668,PVHI,2024-04-28 08:45:14.752000,2024-04-28 08:46:11.753000,0.950017
9,03FIC_1668,PVHI,2024-04-28 08:46:35.753100,2024-04-28 08:47:11.751500,0.599973
